# Use Ollama


In [1]:
# 1. Cài đặt các phụ thuộc (bao gồm zstd) và Ollama
!sudo apt-get update && sudo apt-get install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Kiểm tra GPU
!nvidia-smi

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:5 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:10 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:12 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,546 kB]
Get:13 https://ppa.launchpadcontent.net

In [2]:
import subprocess
import time
import os
import requests

# 1. Tắt các tiến trình cũ
!pkill ollama || true
time.sleep(2)

# 2. Cấu hình môi trường
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["OLLAMA_FLASH_ATTENTION"] = "1"  # Bắt buộc để chạy context dài
os.environ["OLLAMA_NOPRUNE"] = "1"
# 3. Khởi chạy Ollama server ở chế độ nền bằng đường dẫn tuyệt đối
# Sử dụng nohup để đảm bảo tiến trình không bị tắt
print("Đang khởi động Ollama Server...")
process = subprocess.Popen(
    ["/usr/local/bin/ollama", "serve"],
    stdout=open("ollama_stdout.log", "w"),
    stderr=open("ollama_stderr.log", "w"),
    preexec_fn=os.setpgrp
)

# 4. Kiểm tra server cho đến khi sẵn sàng
max_retries = 20
for i in range(max_retries):
    try:
        response = requests.get("http://127.0.0.1:11434/api/tags")
        if response.status_code == 200:
            print("\n[OK] Server Ollama đã khởi động thành công và đang chạy nền!")
            break
    except:
        if i % 5 == 0:
            print(f"Đang chờ server... ({i}/{max_retries})")
        time.sleep(2)
else:
    print("\n[LỖI] Server không phản hồi sau 40 giây. Vui lòng kiểm tra ollama_stderr.log")

Đang khởi động Ollama Server...
Đang chờ server... (0/20)

[OK] Server Ollama đã khởi động thành công và đang chạy nền!


### 3. Tải Model
Tải model trước khi thiết lập kết nối công khai.

In [3]:
import requests
import json

def pull_model(model_name):
    url = "http://127.0.0.1:11434/api/pull"
    payload = {"name": model_name}

    print(f"Đang tải model: {model_name}...")
    try:
        response = requests.post(url, json=payload, stream=True)
        response.raise_for_status()

        for line in response.iter_lines():
            if line:
                data = json.loads(line)
                status = data.get("status", "")
                completed = data.get("completed", 0)
                total = data.get("total", 0)

                if total > 0:
                    percent = completed / total * 100
                    print(f"\rTrạng thái: {status} - {percent:.2f}%", end="")
                else:
                    print(f"\rTrạng thái: {status}", end="")

        print("\n[Xong] Tải model hoàn tất!")
    except Exception as e:
        print(f"\n[Lỗi] Không thể kết nối tới server Ollama: {e}")

pull_model("qwen3.6:35b-a3b")

Đang tải model: qwen3.6:35b-a3b...
Trạng thái: success
[Xong] Tải model hoàn tất!


In [ ]:
# Kiểm tra lại danh sách model thực tế đang có trong server
# 1. Tìm và giết tiến trình đang chiếm port 11434
!sudo fuser -k 11434/tcp
# 2. Đảm bảo các tiến trình ollama khác cũng bị tắt
!pkill -9 ollama
print("Đã giải phóng port 11434. Bây giờ bạn có thể chạy lại cell khởi động server.")
!export OLLAMA_FLASH_ATTENTION=1
!ollama serve

Đã giải phóng port 11434. Bây giờ bạn có thể chạy lại cell khởi động server.
time=2026-04-24T11:31:27.877Z level=INFO source=routes.go:1752 msg="server config" env="map[CUDA_VISIBLE_DEVICES:0 GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MODELS:/root/.ollama/models OLLAMA_MULTIUSER_CACHE:false OLLAMA_NEW_ENGINE:false OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:true OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https://0.0.0.0

# Setup ngrok

In [ ]:
!pip install pyngrok
import os
from pyngrok import ngrok

# Token ngrok
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Xóa tunnel cũ
ngrok.kill()

# Khởi tạo tunnel với header localhost để bypass filter
try:
    # Sử dụng host_header='localhost' và bind_tls=True
    tunnel = ngrok.connect(11434, bind_tls=True, host_header="localhost")
    public_url = tunnel.public_url

    print(f"[THÀNH CÔNG] API Ollama đã mở tại: {public_url}")
    print("\n--- CẤU HÌNH CHO CLAUDE-CLI ---")
    print(f'$env:ANTHROPIC_BASE_URL="{public_url}/v1"')
    print('$env:ANTHROPIC_API_KEY="ollama"')
    print("claude --model qwen3.6:35b-a3b")
    print("\n*Lưu ý: Nếu vẫn bị lỗi, hãy thử mở link trên bằng trình duyệt một lần rồi nhấn 'Visit Site' để xác nhận với ngrok.")
except Exception as e:
    print(f"[LỖI] Ngrok: {e}")

[THÀNH CÔNG] API Ollama đã mở tại: https://gestate-glamour-comfort.ngrok-free.dev

--- CẤU HÌNH CHO CLAUDE-CLI ---
$env:ANTHROPIC_BASE_URL="https://gestate-glamour-comfort.ngrok-free.dev/v1"
$env:ANTHROPIC_API_KEY="ollama"
claude --model qwen3.6:35b-a3b

*Lưu ý: Nếu vẫn bị lỗi, hãy thử mở link trên bằng trình duyệt một lần rồi nhấn 'Visit Site' để xác nhận với ngrok.
